# Stock market chart pattern recognition using deep learning 


In [ ]:
pip install settrade_v2
pip install cassandra-driver
pip install schedule

Note: you may need to restart the kernel to use updated packages.


# 1. Collect Data(Settrade APi)

In [22]:
from settrade_v2 import Investor
import pandas as pd
import cassandra
import re
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from cassandra.cluster import Cluster
import requests
from fuzzywuzzy import fuzz
from collections import defaultdict
from fuzzywuzzy import process
from nltk.tokenize import word_tokenize
import requests
import schedule
import time
from cassandra.cluster import Cluster
from datetime import datetime
import yfinance as yf

In [2]:
cluster = Cluster(['127.0.0.1'])  # or 'localhost'
session = cluster.connect()

In [3]:
# กำหนดค่า API Credentials
investor = Investor(
    app_id="a0iOat7M4FrOrGxS",          # เปลี่ยนเป็น App ID ของคุณ
    app_secret="AJcSTgM8JP+N2Uzi6eBGFxnv7i0T6RPzTX+6FiTgfgnp",  # เปลี่ยนเป็น App Secret ของคุณ
    broker_id="SANDBOX",  # เปลี่ยนเป็น Broker ID ของคุณ
    app_code="SANDBOX"
    
)
market = investor.MarketData()


In [17]:
# กรณี Investor
market = investor.MarketData()


res = market.get_candlestick(
    symbol="TRUBB",
    interval="1d",
    limit=1,
    normalized=True,
)

print(res)
print(type(res))

{'lastSequence': 0, 'time': [1742317200], 'open': [0.68], 'high': [0.73], 'low': [0.68], 'close': [0.71], 'volume': [1419257], 'value': [1004435.12]}
<class 'dict'>


In [40]:
# 👉 เลือกใช้ Keyspace
session.set_keyspace('stock_data')

# 👉 ตรวจสอบและสร้างตาราง (ถ้ายังไม่มี)
session.execute("""
    CREATE TABLE IF NOT EXISTS candlestick_data (
        symbol TEXT,
        time TIMESTAMP PRIMARY KEY,
        open_price FLOAT,
        high_price FLOAT,
        low_price FLOAT,
        close_price FLOAT,
        volume INT,
        value FLOAT
    )
""")
print("✅ Keyspace และ Table พร้อมใช้งาน!")

✅ Keyspace และ Table พร้อมใช้งาน!


In [41]:
# 👉 ฟังก์ชันดึงข้อมูลหุ้น
def fetch_and_store_stock(symbol="TRUBB", interval="1d", limit=10):
    res = market.get_candlestick(symbol=symbol, interval=interval, limit=limit, normalized=True)

    if not res:
        print(f"⚠️ ไม่พบข้อมูลสำหรับ {symbol}")
        return

    for i in range(len(res["time"])):
        timestamp = datetime.fromtimestamp(res["time"][i])  
        open_price = res["open"][i]
        high_price = res["high"][i]
        low_price = res["low"][i]
        close_price = res["close"][i]
        volume = res["volume"][i]
        value = res["value"][i]

        session.execute("""
            INSERT INTO candlestick_data (symbol, time, open_price, high_price, low_price, close_price, volume, value)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (symbol, timestamp, open_price, high_price, low_price, close_price, volume, value))

    print(f"✅ เพิ่มข้อมูล {len(res['time'])} รายการของหุ้น {symbol} สำเร็จ!")

# 👉 ทดสอบดึงข้อมูลหุ้น BBL
fetch_and_store_stock(symbol="TRUBB", interval="1d", limit=10)

✅ เพิ่มข้อมูล 10 รายการของหุ้น TRUBB สำเร็จ!


In [45]:
def insert_candlestick_data(symbol, interval="1d", limit=10):
    try:
        # 👉 ดึงข้อมูลจาก Settrade API
        res = market.get_candlestick(symbol=symbol, interval=interval, limit=limit, normalized=True)

        if not res:
            print(f"⚠️ ไม่พบข้อมูลสำหรับ {symbol}")
            return

        # 👉 เชื่อมต่อกับ Cassandra
        cluster = Cluster(['127.0.0.1'])  
        session = cluster.connect()
        session.set_keyspace('stock_data')

        # 👉 บันทึกข้อมูลลงฐานข้อมูล
        for i in range(len(res["time"])):
            timestamp = datetime.fromtimestamp(res["time"][i])
            open_price = res["open"][i]
            high_price = res["high"][i]
            low_price = res["low"][i]
            close_price = res["close"][i]
            volume = res["volume"][i]
            value = res["value"][i]

            session.execute("""
                INSERT INTO candlestick_data (symbol, time, open_price, high_price, low_price, close_price, volume, value)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            """, (symbol, timestamp, open_price, high_price, low_price, close_price, volume, value))

        print(f"✅ เพิ่มข้อมูล {len(res['time'])} รายการของหุ้น {symbol} สำเร็จ!")

    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาดในการดึงข้อมูลของหุ้น {symbol}: {e}")


In [32]:
# 🔄 รายชื่อหุ้นที่ต้องการทดสอบ
symbols = ["TRUBB"]
intervals = ["1d"]

# ✅ วนลูปดึงข้อมูลหุ้นตามช่วงเวลาที่กำหนด
for symbol in symbols:
    for interval in intervals:
        print(f"🔄 กำลังดึงข้อมูลสำหรับ {symbol} (Interval: {interval})...")
        insert_candlestick_data(symbol, interval=interval, limit=10)
        time.sleep(2)  # ⏳ ลด API Rate Limit


🔄 กำลังดึงข้อมูลสำหรับ TRUBB (Interval: 1d)...
✅ เพิ่มข้อมูล 10 รายการของหุ้น TRUBB สำเร็จ!


In [ ]:
# สั่งเลือกข้อมูลจากตาราง candlestick_data
symbol = "TRUBB"
query = f"SELECT time, open_price, high_price, low_price, close_price FROM candlestick_data WHERE symbol = '{symbol}' ALLOW FILTERING;"

# ดึงข้อมูลจาก Cassandra
rows = session.execute(query)

# สร้าง DataFrame จากผลลัพธ์
data = []
for row in rows:
    data.append({
        "time": row.time,
        "close_price": row.close_price
    })

df = pd.DataFrame(data)

# แสดง DataFrame
print(df)

         time  close_price
0  2024-12-19         1.01
1  2024-10-29         1.11
2  2024-10-04         1.16
3  2024-11-06         1.10
4  2024-11-07         1.09
..        ...          ...
95 2024-10-28         1.09
96 2024-10-10         1.17
97 2024-12-25         1.04
98 2025-03-18         0.69
99 2024-12-17         1.01

[100 rows x 2 columns]
        time  close_price
0 2024-12-19         1.01
1 2024-10-29         1.11
2 2024-10-04         1.16
3 2024-11-06         1.10
4 2024-11-07         1.09


In [39]:
import matplotlib.pyplot as plt
import pandas as pd

# ตรวจสอบว่า df มีข้อมูลหรือไม่
if df.empty:
    print("⚠️ ไม่มีข้อมูลใน DataFrame")
else:
    # แปลงคอลัมน์ 'time' ให้เป็น datetime
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)

    # สร้างกราฟ
    plt.figure(figsize=(12, 6))
    plt.plot(df.index, df['close_price'], marker='o', linestyle='-', color='b', label='Close Price')

    # ตั้งค่ากราฟ
    plt.title(f"Stock Price of {symbol}")
    plt.xlabel("Date")
    plt.ylabel("Close Price (THB)")
    plt.legend()
    plt.grid()

    # แสดงกราฟ
    plt.show()



KeyError: 'time'